# Lakehouse Views Creation Pipeline - Generic & On-Demand

Dedicated on-demand pipeline to create persistent and temporary Spark SQL / Unity Catalog views across Bronze, Silver, and Gold layers.

### Supported View Definition Modes:
1. **Table-Based Views (`definition_type: "table"`)**: Standard projections with column selection and `WHERE` filtering on an underlying Delta table.
2. **Custom SQL Query Views (`definition_type: "query"`)**: Fully customized SQL statements (joins, window functions, CTEs, custom business logic, aggregations) with automatic environment catalog/schema token interpolation (e.g. `{catalog}`, `{silver_schema}`, `{gold_schema}`).

This pipeline runs **on-demand** or as an explicit workflow step when view definitions are modified, decoupling view creation from ingestion/aggregation runs.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "staging", "prod"], "Environment / Stage")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.dropdown("layer", "all", ["all", "bronze", "silver", "gold"], "Target Layer")
dbutils.widgets.text("target_view_name", "", "Specific View Name (Optional, blank for all)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

try:
    _stage = dbutils.widgets.get("stage").strip()
except Exception:
    _stage = ""
environment = _stage or dbutils.widgets.get("environment")
client_id = dbutils.widgets.get("client_id").strip() or None
layer = dbutils.widgets.get("layer").strip().lower()
target_view_name = dbutils.widgets.get("target_view_name").strip() or None

print(f"Executing Views Creation Pipeline:")
print(f"  - Environment:  {environment}")
print(f"  - Client Scope: {client_id or 'Global (All Clients)'}")
print(f"  - Layer Filter: {layer}")
print(f"  - Target View:  {target_view_name or 'All Configured Views'}")

In [ ]:
# Imports & Path setup
import sys

from lakeforge.metadata.config_parser import ConfigParser
from lakeforge.views.view_manager import create_view_manager
print("✅ ViewManager and ConfigParser loaded successfully")

In [ ]:
# Retrieve environment metadata
env_config = ConfigParser.get_environment_config(config_dir, environment)

# Load view configurations with multi-tenant client resolution
all_view_configs = ConfigParser.get_view_configs(config_dir, environment, client_id=client_id)

# Apply layer and name filters
selected_views = []
for vc in all_view_configs:
    if target_view_name and vc.view_name != target_view_name:
        continue
    
    if layer != "all":
        expected_schema = env_config.get(f"{layer}_schema")
        if vc.schema != expected_schema and layer not in vc.schema.lower():
            continue
            
    selected_views.append(vc)

print(f"📋 Selected {len(selected_views)} view(s) to process:")
for v in selected_views:
    if v.definition_type == "query":
        print(f"  * [CUSTOM QUERY] {v.catalog}.{v.schema}.{v.view_name}")
        print(f"    SQL: {v.query}")
    else:
        print(f"  * [TABLE PROJECTION] {v.catalog}.{v.schema}.{v.view_name} (from {v.source_table})")

In [ ]:
# Materialize views in Spark / Unity Catalog
if not selected_views:
    print("⚠️ No matching views found for the specified criteria.")
else:
    view_manager = create_view_manager(spark, env_config)
    success_count = 0
    failure_count = 0
    
    for vc in selected_views:
        try:
            view_manager.create_view(vc)
            print(f"✅ Successfully created view: {vc.catalog}.{vc.schema}.{vc.view_name}")
            success_count += 1
        except Exception as e:
            print(f"❌ Failed to create view {vc.view_name}: {str(e)}")
            failure_count += 1
            
    print("=" * 60)
    print(f"📊 VIEW CREATION SUMMARY: {success_count} succeeded, {failure_count} failed")
    print("=" * 60)
    if failure_count > 0:
        raise RuntimeError(f"{failure_count} view(s) failed during creation.")